In [ ]:
# Final project (Hyperlocal News Anomaly Detection and Source Attribution with Application Hosted on GCP/AWS)
import pandas as pd
import spacy
from textblob import TextBlob
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer

# 1. Load Data
path = "Articles.csv"
df = pd.read_csv(path, encoding='latin1', on_bad_lines='skip', engine='python').dropna(subset=['Article'])
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

# 2. Sentiment, Heading & NewsType
df['Article_Clean'] = df['Article'].str.lower()
df['Sentiment'] = df['Article_Clean'].apply(lambda x: TextBlob(x).sentiment.polarity)
df['Emotion'] = df['Article_Clean'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
df['Heading'] = df['Article'].str[:50] + "..."
df['NewsType'] = df['Article_Clean'].apply(lambda x: "Breaking" if any(k in x for k in ["urgent", "just in"]) else "General")

# 3. Fast NER & Topic Modeling (LDA)
df['Locations'] = [[e.text for e in d.ents if e.label_ in ('GPE', 'LOC')] for d in nlp.pipe(df['Article'], batch_size=100)]
vec = CountVectorizer(stop_words='english', max_features=1000)
dtm = vec.fit_transform(df['Article_Clean'])
lda = LatentDirichletAllocation(n_components=5, random_state=42)
df['Topic_ID'] = lda.fit_transform(dtm).argmax(axis=1)

# 4. Temporal Engineering
if 'Date' in df.columns:
    dt = pd.to_datetime(df['Date'])
    df = df.assign(Year=dt.dt.year, Month=dt.dt.month, Day=dt.dt.day, Time=dt.dt.time, Is_Weekend=dt.dt.dayofweek > 4)

# 5. Anomaly Detection
df['Is_Anomaly'] = (df['Sentiment'] - df['Sentiment'].mean()).abs() > (2 * df['Sentiment'].std())

# Save
df.to_csv("Enriched_Articles.csv", index=False)
print("Project Data Enriched successfully!")

Project Data Enriched successfully!


In [ ]:
# sentence transformer (BERT)
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Load your local CSV file
file_path = "/content/Articles.csv"
df = pd.read_csv(file_path, encoding='latin1')

# 2. Select the column containing your text
text_data = df['Article'].astype(str).tolist()

# 3. Initialize the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 4. Generate embeddings
embeddings = model.encode(text_data, show_progress_bar=True)

print(f"Generated {len(embeddings)} embeddings with dimension {embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Generated 2692 embeddings with dimension 384


In [ ]:
# anamoly detection (Linguistic Anomaly Detection)

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest

# File path
file_path = "/content/Articles.csv"

# Load the file path
df = pd.read_csv(file_path, encoding='latin1')
text_col = 'Article'

# 1. Vectorize text (TF-IDF)
vec = TfidfVectorizer(stop_words='english', max_features=500)
X = vec.fit_transform(df[text_col].fillna(''))

# 2. Detect Anomalies (Isolation Forest)
model = IsolationForest(contamination=0.05, random_state=42)
df['is_anomaly'] = model.fit_predict(X.toarray())

# 3. Output result
anomalies = df[df['is_anomaly'] == -1]
print(f"Found {len(anomalies)} anomalies:")
print(anomalies[text_col].head())

Found 135 anomalies:
4     NEW YORK: US oil prices Monday slipped below $...
24    Hong Kong: The euro extended its gains against...
25    New York: Oil prices rebounded Tuesday from si...
48    Singapore: Oil prices edged higher in Asia Thu...
55    New York: Oil prices fell on Wednesday before ...
Name: Article, dtype: object
